# 16. Advanced Time-Series & Intervals: Beginner Guide

### 📌 Overview & Architectural Context
Welcome to **16. Advanced Time-Series & Intervals**. Quantitative finance and market telemetry require precise calendar awareness and interval discretization. This notebook explores business day sequences (`bdate_range()`, `pd.offsets.BDay`), trading session market hour filters (`at_time()`, `between_time()`), closest prior timestamp lookups (`asof()`), offset settlement arithmetic ($T+2$), and equal-frequency quantile binning (`qcut()`).

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Datetime Ranges: `pd.date_range()`
- [x] 🔹 Accounting Periods: `pd.period_range()`
- [x] 🔹 Mathematical Intervals: `pd.interval_range()`
- [x] 🔹 Business Day Calendars: `pd.bdate_range()`
- [x] 🔹 Equal-Width Discretization: `pd.cut()`
- [x] 🔹 Equal-Frequency Quantile Binning: `pd.qcut()`


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns


### 🔹 Datetime Ranges: `pd.date_range()`
- **What it does:** Generates a fixed-frequency DatetimeIndex with evenly spaced calendar or time timestamps between bounds or over a specified length.
- **Syntax:** `pd.date_range()`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** You must specify exactly two of `start`, `end`, and `periods`. If `freq` is omitted, the default frequency is calendar daily (`'D'`).
- **Dataset Application & Code Demonstration:** Demonstrates Datetime Ranges with practical fintech data structures and variables in the following code block.


In [2]:
settlement_dates = pd.date_range(start='2025-01-01', periods=7, freq='D')
print('Generated Settlement Dates:\n', settlement_dates)

Generated Settlement Dates:
 DatetimeIndex(['2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04',
               '2025-01-05', '2025-01-06', '2025-01-07'],
              dtype='datetime64[ns]', freq='D')


### 🔹 Accounting Periods: `pd.period_range()`
- **What it does:** Generates a fixed-frequency PeriodIndex representing regular time intervals (spans of time such as months, quarters, or years) rather than point-in-time timestamps.
- **Syntax:** `pd.period_range()`
  - **Parameters:**
    - `value` (*object*): The target value to count occurrences of.
- **Key Note:** Periods represent intervals (durations) rather than instantaneous points in time. Slicing with PeriodIndex naturally binds all sub-events occurring within that time span.
- **Dataset Application & Code Demonstration:** Demonstrates Accounting Periods with practical fintech data structures and variables in the following code block.


In [3]:
fiscal_periods = pd.period_range(start='2025-01', periods=6, freq='M')
print('Fiscal Period Spans:\n', fiscal_periods)

Fiscal Period Spans:
 PeriodIndex(['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06'], dtype='period[M]')


### 🔹 Mathematical Intervals: `pd.interval_range()`
- **What it does:** Generates an IntervalIndex of bounded numeric, datetime, or timedelta ranges with consistent step sizes and boundary closures.
- **Syntax:** `pd.interval_range()`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Intervals are ideal for binning continuous variables (e.g., age brackets, spend tiers) into non-overlapping mathematical domains.
- **Dataset Application & Code Demonstration:** Demonstrates Mathematical Intervals with practical fintech data structures and variables in the following code block.


In [4]:
spend_intervals = pd.interval_range(start=0, end=1500, periods=5, closed='left')
print('Spend Intervals:\n', spend_intervals)

Spend Intervals:
 IntervalIndex([[0, 300), [300, 600), [600, 900), [900, 1200), [1200, 1500)], dtype='interval[int64, left]')


### 🔹 Business Day Calendars: `pd.bdate_range()`
- **What it does:** Generates a fixed-frequency DatetimeIndex strictly constrained to standard business/working days (Monday through Friday), omitting weekends and optional custom holidays.
- **Syntax:** `pd.bdate_range()`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Use `holidays` and `weekmask` to model regional financial trading calendars (e.g., Sunday–Thursday workweeks in the Middle East).
- **Dataset Application & Code Demonstration:** Demonstrates Business Day Calendars with practical fintech data structures and variables in the following code block.


In [5]:
banking_days = pd.bdate_range(start='2025-01-03', periods=5, freq='B')
print('Banking Business Days (skips weekends):\n', banking_days)

Banking Business Days (skips weekends):
 DatetimeIndex(['2025-01-03', '2025-01-06', '2025-01-07', '2025-01-08',
               '2025-01-09'],
              dtype='datetime64[ns]', freq='B')


### 🔹 Equal-Width Discretization: `pd.cut()`
- **What it does:** Binds values of continuous numerical arrays into discrete categorical bins of equal width or along user-defined boundary edges.
- **Syntax:** `pd.cut()`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Equal-width binning (`pd.cut`) divides the data span `(max - min)` into equal ranges, regardless of how data points are distributed across the spectrum.
- **Dataset Application & Code Demonstration:** Applies Equal-Width Discretization on fintech records using columns `transaction_amount` to demonstrate real-world execution.


In [6]:
df_valid = df.dropna(subset=['transaction_amount']).copy()
df_valid['spend_tier'] = pd.cut(df_valid['transaction_amount'], bins=[0, 50, 200, 800, 5000], labels=['Micro', 'Small', 'Medium', 'Large'])
print('Discretized Spend Tier Breakdown:\n', df_valid['spend_tier'].value_counts())

Discretized Spend Tier Breakdown:
 spend_tier
Large     8551
Medium    4316
Small     1073
Micro      311
Name: count, dtype: int64


### 🔹 Equal-Frequency Quantile Binning: `pd.qcut()`
- **What it does:** Discretizes numerical data into categorical buckets based on sample quantiles, ensuring each resulting bin contains an approximately equal number of observations.
- **Syntax:** `pd.qcut()`
- **Key Note:** Unlike `pd.cut()`, `pd.qcut()` constructs bins with equal observation counts, preventing skewed distributions from creating empty or sparse bins.
- **Dataset Application & Code Demonstration:** Applies Equal-Frequency Quantile Binning on fintech records using columns `transaction_amount` to demonstrate real-world execution.


In [7]:
df_valid['spend_quartile'] = pd.qcut(df_valid['transaction_amount'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
print('Balanced Quartile Counts:\n', df_valid['spend_quartile'].value_counts())

Balanced Quartile Counts:
 spend_quartile
Q1    3563
Q2    3563
Q4    3563
Q3    3562
Name: count, dtype: int64


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Fraud Rate Distribution Across Spend Quartiles
- **Objective:** Q1: Fraud Rate Distribution Across Spend Quartiles
- **Approach:** Analyze whether the fraud rate increases exponentially in the highest spend quartile (Q4).
- **Syntax:** `df.groupby('spend_quartile')['is_fraud'].agg(['count', 'mean'])`

In [8]:
quartile_fraud = df_valid.groupby('spend_quartile', observed=False)['is_fraud'].agg(['count', 'mean'])
print('Fraud Rate Breakdown Across Spend Quartiles:\n', quartile_fraud)

Fraud Rate Breakdown Across Spend Quartiles:
                 count      mean
spend_quartile                 
Q1               3563  0.011507
Q2               3563  0.007578
Q3               3562  0.009264
Q4               3563  0.423800
